# One CSP asset charging TES and BES

A standalone demonstration of one thermal source feeding two storage media with independent conversion efficiencies, rates, and costs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
while not (root / 'enliten').is_dir():
    if root.parent == root: raise RuntimeError('Run from inside the ENLITEN repository.')
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from enliten import ChargingPath, Generation, LCOECalculator, Site, Storage, System
data_dir = root / 'examples' / 'data'

def profile(filename):
    frame = pd.read_csv(data_dir / filename)
    return pd.Series(frame['PNM'].to_numpy(float), index=pd.to_datetime(frame.iloc[:, 0], utc=True))

demand_full, csp_full = profile('PNM_demand.csv'), profile('PNM_csp_th_av.csv')
hours, start = 24 * 14, pd.Timestamp('2023-03-01', tz='UTC')
window = demand_full.index.get_loc(start)
load = (demand_full.iloc[window:window + hours] * 0.05).rename('load_MW')

site = Site('microgrid')
csp_tes_capex = 150.0 * 1_000 * 7_912
bes_capex = 750.0 * 1_000 * 300
csp = Generation('csp', site, csp_full.iloc[window:window + hours], 'thermal', False, False, capex=csp_tes_capex, opex=150.0 * 1_000 * 74.6)
tes = Storage('tes', site, 2_500.0, 150.0, 'thermal', 'electric', 0.50, maximum_stored_energy_rate_MW=300.0, variable_opex_USD_per_MWh=3.8)
bes = Storage('bes', site, 750.0, 150.0, 'electric', 'electric', 0.90, maximum_stored_energy_rate_MW=150.0, capex=bes_capex, opex=0.025 * bes_capex)
paths = [ChargingPath('csp', 'tes', 'thermal', 'thermal', 0.90, 200.0, priority=0), ChargingPath('csp', 'bes', 'thermal', 'electric', 0.40, 150.0 / 0.40, priority=1)]
system = System(load, [tes, bes, csp], paths)
system.timeseries.filter(regex='csp_to_|tes_MWh|bes_MWh').head()

In [ ]:
system.operation_metrics()

In [ ]:
LCOECalculator.from_system(system).calculate_lcoe_metrics()

In [ ]:
fig, ax = system.plot_storage_capacity(start_date=0, days=2)
fig